# Geometry-MMALS G1 v1.0.3
## Cross-Angle Paired Functional Geometry

**Scientific status:** protocol implementation and development experiment.  
**Accepted claim:** C0 pipeline/protocol integrity only.  
**Not claimed:** C1-C6 qualification, quantum advantage, backward transfer, or domain-general geometry.

This notebook corrects the main scientific limitation of v1.0.2: each geometry
batch previously contained one fixed angle, so the geometry loss never observed
two distinct context values. v1.0.3 groups multiple rotated views of the **same
source image**, making route-distance versus angle-distance a real trainable and
measurable relation.

## Research history and change tracking

| Version | Question | Outcome |
|---|---|---|
| v1.0.0 | Can functional MMALS geometry be specified and falsified? | Article, protocol and C0 scaffold created. |
| v1.0.1 | Can release claims, gates, memory stubs and notation be audited? | C0-only status and pilot gates formalized. |
| v1.0.2 | Can the Fisher-Rao route loss and Colab pipeline run without NaNs? | Numerical stability passed; finite traces obtained. |
| **v1.0.3** | Does the optimization unit actually contain cross-angle evidence? | Same-source paired protocol introduced. |

### Tracked changes

- **CHG-103-01:** same-source multi-angle data primitive.
- **CHG-103-02:** paired square-root-simplex geometry loss.
- **CHG-103-03:** continual stages use only angles already seen.
- **CHG-103-04:** source index retained in every trace.
- **CHG-103-05:** source-block bootstrap and factor-centroid metrics.
- **CHG-103-06:** held-out interpolation accuracy, NLL and route/context interpolation.
- **CHG-103-07:** staged accuracy matrix and forgetting.
- **CHG-103-08:** signed tangent effect with matched orthogonal controls.
- **CHG-103-09:** package version and git SHA recorded in the manifest.
- **CHG-103-10:** explicit non-claims and qualification checklist.

## Why the v1.0.2 run was non-diagnostic

The v1.0.2 training loop used one `FixedAngleMNIST` loader at a time. Inside a
batch, every factor value was identical:

\[
u_1=u_2=\cdots=u_B.
\]

The route loss could enforce within-angle consistency, but it could not learn an
ordered relation among `-60, -30, 0, 30, 60` degrees. Global all-sample
pairwise metrics also mixed digit identity with rotation and produced
dependence-inflated p-values and tie-fragile kNN scores.

v1.0.3 changes the optimization and measurement unit to:

> one source image observed under several declared angles.

This is a stronger test of geometry, while still remaining a supervised
grounded G1-A variant.

In [ ]:
import os
import sys
import shutil
import subprocess
import importlib
import importlib.metadata
from pathlib import Path

REPO_URL = "https://github.com/gharbonnier78/geometry-mmalls-g1.git"
REPO_DIR = Path("/content/geometry-mmalls-g1")
SRC_DIR = REPO_DIR / "src"
EXPECTED_VERSION = "1.0.3"
FORCE_REFRESH = True
STRICT_VERSION = True

if FORCE_REFRESH and REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        check=True,
    )

os.chdir(REPO_DIR)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(REPO_DIR)],
    check=True,
)

src_path = str(SRC_DIR)
if src_path not in sys.path:
    sys.path.insert(0, src_path)
importlib.invalidate_caches()

import geometry_mmalls

package_version = importlib.metadata.version("geometry-mmalls-g1")
git_sha = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True
).strip()

print("Python:", sys.executable)
print("Package:", geometry_mmalls.__file__)
print("Package version:", package_version)
print("Git SHA:", git_sha)

if package_version != EXPECTED_VERSION:
    message = (
        f"Expected geometry-mmalls-g1 {EXPECTED_VERSION}, got {package_version}. "
        "Push the v1.0.3 package or install the supplied v1.0.3 ZIP before running."
    )
    if STRICT_VERSION:
        raise RuntimeError(message)
    print("WARNING:", message)

In [ ]:
from pathlib import Path
import copy
import json
import math
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import yaml
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import MNIST

from geometry_mmalls.data import FixedAngleMNIST, MultiAngleMNIST
from geometry_mmalls.geometry import (
    fisher_rao_distance,
    paired_route_geometry_loss,
)
from geometry_mmalls.metrics import (
    bootstrap_mean_ci,
    centroid_geometry_scores,
    grouped_geometry_scores,
)
from geometry_mmalls.model import GeometryMMALS, SmallConvEncoder

config = yaml.safe_load(Path("configs/rotated_mnist_g1.yaml").read_text())

RUN_MODE = "development"  # "development" or "qualification"
SEED = 0
METHODS = ["no_geometry", "paired_geometry"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

train_angles = list(map(float, config["data"]["train_angles"]))
interp_angles = list(map(float, config["data"]["interpolation_angles"]))
extra_angles = list(map(float, config["data"]["extrapolation_angles"]))
all_eval_angles = train_angles + interp_angles + extra_angles

if RUN_MODE == "development":
    TRAIN_SOURCE_LIMIT = int(config["paired_protocol"]["development_source_limit"])
    TEST_SOURCE_LIMIT = int(config["paired_protocol"]["development_test_source_limit"])
    SENSORY_EPOCHS = 2
    STAGE_EPOCHS = 2
    BOOTSTRAP_SAMPLES = 500
else:
    TRAIN_SOURCE_LIMIT = int(config["paired_protocol"]["full_source_limit"])
    TEST_SOURCE_LIMIT = 2000
    SENSORY_EPOCHS = int(config["training"]["sensory_pretrain_epochs"])
    STAGE_EPOCHS = int(config["training"]["stage_epochs"])
    BOOTSTRAP_SAMPLES = int(config["metrics"]["bootstrap_samples"])

SOURCE_BATCH_SIZE = int(config["paired_protocol"]["source_batch_size"])
print({
    "run_mode": RUN_MODE,
    "device": str(DEVICE),
    "methods": METHODS,
    "train_source_limit": TRAIN_SOURCE_LIMIT,
    "test_source_limit": TEST_SOURCE_LIMIT,
})

## 1. Numerical and structural self-test

The training loss uses chordal distance in square-root simplex coordinates.
This is monotonic with Fisher-Rao distance but avoids the derivative singularity
of `arccos` during optimization. Fisher-Rao remains the evaluation distance.

In [ ]:
_probe_logits = torch.randn(8, 5, 4, requires_grad=True)
_probe_routes = torch.softmax(_probe_logits, dim=-1)
_probe_angles = torch.tensor([-60.0, -30.0, 0.0, 30.0, 60.0])
_probe_loss = paired_route_geometry_loss(_probe_routes, _probe_angles)
_probe_loss.backward()

assert torch.isfinite(_probe_loss)
assert _probe_logits.grad is not None
assert torch.isfinite(_probe_logits.grad).all()
print("Paired route-geometry backward gate: PASS", float(_probe_loss.detach()))

## 2. Controlled source split

The same source indices are used at every angle. Interpolation and extrapolation
angles are never included in training. The split is deterministic and recorded.

In [ ]:
root = Path(config["data"]["root"])
base_train = MNIST(root=str(root), train=True, download=True)
base_test = MNIST(root=str(root), train=False, download=True)

rng = np.random.default_rng(SEED)
train_indices = rng.permutation(len(base_train))[:TRAIN_SOURCE_LIMIT].tolist()
test_indices = rng.permutation(len(base_test))[:TEST_SOURCE_LIMIT].tolist()

def fixed_loader(angle, train, indices, shuffle):
    ds = FixedAngleMNIST(root, angle=angle, train=train, download=True)
    ds = Subset(ds, indices)
    return DataLoader(
        ds,
        batch_size=128,
        shuffle=shuffle,
        num_workers=0,
    )

def multi_loader(angles, train, indices, shuffle):
    ds = MultiAngleMNIST(
        root,
        angles=angles,
        train=train,
        indices=indices,
        download=True,
    )
    return DataLoader(
        ds,
        batch_size=SOURCE_BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,
    )

split_manifest = {
    "seed": SEED,
    "train_source_count": len(train_indices),
    "test_source_count": len(test_indices),
    "train_index_checksum": int(np.sum(train_indices)),
    "test_index_checksum": int(np.sum(test_indices)),
}
split_manifest

## 3. Pretrain and freeze the sensory grove

The frozen encoder is a control boundary. Geometry already present in `z0` is
reported alongside context, route and synthesis geometry.

In [ ]:
latent_dim = int(config["model"]["latent_dim"])
encoder = SmallConvEncoder(latent_dim).to(DEVICE)
sensory_head = torch.nn.Linear(latent_dim, 10).to(DEVICE)
sensory_optimizer = torch.optim.AdamW(
    list(encoder.parameters()) + list(sensory_head.parameters()),
    lr=float(config["training"]["learning_rate"]),
    weight_decay=float(config["training"]["weight_decay"]),
)

sensory_history = []
sensory_loader = fixed_loader(0.0, True, train_indices, True)

for epoch in range(SENSORY_EPOCHS):
    encoder.train()
    total = correct = 0
    loss_sum = 0.0
    for x, y, _, _ in sensory_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = sensory_head(encoder(x))
        loss = F.cross_entropy(logits, y)
        if not torch.isfinite(loss):
            raise FloatingPointError("Non-finite sensory loss")
        sensory_optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(encoder.parameters()) + list(sensory_head.parameters()), 5.0
        )
        sensory_optimizer.step()
        total += y.numel()
        correct += (logits.argmax(1) == y).sum().item()
        loss_sum += float(loss.detach()) * y.numel()
    row = {
        "epoch": epoch,
        "loss": loss_sum / total,
        "accuracy": correct / total,
    }
    sensory_history.append(row)
    print(row)

sensory_state = copy.deepcopy(encoder.state_dict())

## 4. Continual training variants

### `no_geometry`
Same architecture and sequential angle schedule, but no paired route geometry.

### `paired_geometry`
At stage `t`, each source is rendered at all angles already seen in stages
`0..t`. Classification loss is applied to the current angle only. Previous
views are geometry anchors. This is not claimed as a replay-free continual
learning method; it is a controlled G1-A geometry test.

In [ ]:
def host_diversity(host_outputs):
    h = F.normalize(host_outputs, dim=-1)
    sim = torch.einsum("bhd,bjd->bhj", h, h)
    mask = ~torch.eye(sim.shape[-1], dtype=torch.bool, device=sim.device)
    return sim[:, mask].square().mean()

def assert_finite_trace(trace, where):
    tensors = {
        "z0": trace.z0,
        "context": trace.context,
        "route": trace.route,
        "host_outputs": trace.host_outputs,
        "z_mm": trace.z_mm,
        "logits": trace.logits,
    }
    bad = [name for name, value in tensors.items() if not torch.isfinite(value).all()]
    if bad:
        raise FloatingPointError(f"Non-finite tensors at {where}: {bad}")

@torch.no_grad()
def evaluate_model(model, angles, stage, method):
    rows = []
    model.eval()
    for angle in angles:
        loader = fixed_loader(angle, False, test_indices, False)
        total = correct = 0
        nll_sum = 0.0
        for x, y, _, _ in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            trace = model(x)
            assert_finite_trace(trace, f"eval method={method}, angle={angle}")
            nll = F.cross_entropy(trace.logits, y, reduction="sum")
            total += y.numel()
            correct += (trace.logits.argmax(1) == y).sum().item()
            nll_sum += float(nll)
        rows.append({
            "method": method,
            "stage": stage,
            "angle": angle,
            "angle_type": (
                "train" if angle in train_angles
                else "interpolation" if angle in interp_angles
                else "extrapolation"
            ),
            "accuracy": correct / total,
            "nll": nll_sum / total,
        })
    return rows

def build_model():
    local_encoder = SmallConvEncoder(latent_dim).to(DEVICE)
    local_encoder.load_state_dict(sensory_state)
    return GeometryMMALS(
        local_encoder,
        latent_dim=latent_dim,
        context_dim=int(config["model"]["context_dim"]),
        num_hosts=int(config["model"]["num_hosts"]),
        host_hidden_dim=int(config["model"]["host_hidden_dim"]),
        freeze_encoder=True,
    ).to(DEVICE)

def train_method(method):
    model = build_model()
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=float(config["training"]["learning_rate"]),
        weight_decay=float(config["training"]["weight_decay"]),
    )
    stage_rows = []
    evaluation_rows = []

    for stage, current_angle in enumerate(train_angles):
        seen_angles = train_angles[: stage + 1]
        totals = {"samples": 0, "loss": 0.0, "ce": 0.0, "geo": 0.0, "div": 0.0}

        if method == "paired_geometry":
            loader = multi_loader(seen_angles, True, train_indices, True)
        else:
            loader = fixed_loader(current_angle, True, train_indices, True)

        model.train()
        for epoch in range(STAGE_EPOCHS):
            for batch_id, batch in enumerate(loader):
                if method == "paired_geometry":
                    views, y, factors, _ = batch
                    batch_size, angle_count = views.shape[:2]
                    flat_views = views.reshape(-1, *views.shape[2:]).to(DEVICE)
                    y = y.to(DEVICE)
                    factors = factors.to(DEVICE)

                    trace = model(flat_views)
                    assert_finite_trace(
                        trace,
                        f"{method}, stage={stage}, epoch={epoch}, batch={batch_id}",
                    )
                    logits = trace.logits.reshape(batch_size, angle_count, -1)
                    routes = trace.route.reshape(batch_size, angle_count, -1)
                    hosts = trace.host_outputs.reshape(
                        batch_size, angle_count, trace.host_outputs.shape[1], -1
                    )

                    current_index = angle_count - 1
                    ce = F.cross_entropy(logits[:, current_index], y)
                    geo = paired_route_geometry_loss(
                        routes,
                        factors,
                        bandwidth=float(config["losses"]["route_bandwidth_degrees"]),
                        far_margin=float(config["losses"]["paired_route_far_margin"]),
                        far_weight=float(config["losses"]["paired_route_far_weight"]),
                        match_weight=float(config["losses"]["paired_route_match_weight"]),
                    )
                    div = host_diversity(hosts[:, current_index])
                    sample_count = batch_size
                else:
                    x, y, _, _ = batch
                    x, y = x.to(DEVICE), y.to(DEVICE)
                    trace = model(x)
                    assert_finite_trace(
                        trace,
                        f"{method}, stage={stage}, epoch={epoch}, batch={batch_id}",
                    )
                    ce = F.cross_entropy(trace.logits, y)
                    geo = trace.route.sum() * 0.0
                    div = host_diversity(trace.host_outputs)
                    sample_count = y.numel()

                loss = (
                    float(config["losses"]["classification"]) * ce
                    + float(config["losses"]["route_geometry"]) * geo
                    + float(config["losses"]["host_functional_diversity"]) * div
                )
                if not torch.isfinite(torch.stack([ce, geo, div, loss])).all():
                    raise FloatingPointError(
                        f"Non-finite loss in {method}, stage={stage}: "
                        f"ce={ce.item()}, geo={geo.item()}, div={div.item()}"
                    )

                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                grad_norm = torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], 5.0
                )
                if not torch.isfinite(torch.as_tensor(grad_norm)):
                    raise FloatingPointError("Non-finite gradient norm")
                optimizer.step()

                totals["samples"] += sample_count
                totals["loss"] += float(loss.detach()) * sample_count
                totals["ce"] += float(ce.detach()) * sample_count
                totals["geo"] += float(geo.detach()) * sample_count
                totals["div"] += float(div.detach()) * sample_count

        count = max(totals.pop("samples"), 1)
        stage_row = {
            "method": method,
            "stage": stage,
            "current_angle": current_angle,
            "seen_angles": json.dumps(seen_angles),
            **{key: value / count for key, value in totals.items()},
        }
        stage_rows.append(stage_row)
        print(stage_row)
        evaluation_rows.extend(evaluate_model(model, all_eval_angles, stage, method))

    return model, pd.DataFrame(stage_rows), pd.DataFrame(evaluation_rows)

In [ ]:
models = {}
stage_tables = []
evaluation_tables = []

for method in METHODS:
    print("\n=== TRAINING", method, "===")
    model, stage_df, evaluation_df = train_method(method)
    models[method] = model
    stage_tables.append(stage_df)
    evaluation_tables.append(evaluation_df)

stage_df = pd.concat(stage_tables, ignore_index=True)
evaluation_df = pd.concat(evaluation_tables, ignore_index=True)
stage_df

## 5. Staged accuracy and forgetting

Forgetting is computed only on trained angles. Held-out angles are evaluated for
interpolation and extrapolation but are never considered learned tasks.

In [ ]:
def forgetting_table(evaluation):
    rows = []
    final_stage = int(evaluation.stage.max())
    for method in evaluation.method.unique():
        sub = evaluation[
            (evaluation.method == method)
            & (evaluation.angle_type == "train")
        ]
        for angle in train_angles:
            angle_rows = sub[sub.angle == angle].sort_values("stage")
            seen_stage = train_angles.index(angle)
            angle_rows = angle_rows[angle_rows.stage >= seen_stage]
            best = float(angle_rows.accuracy.max())
            final = float(angle_rows[angle_rows.stage == final_stage].accuracy.iloc[0])
            rows.append({
                "method": method,
                "angle": angle,
                "best_accuracy": best,
                "final_accuracy": final,
                "forgetting": best - final,
            })
    return pd.DataFrame(rows)

forgetting_df = forgetting_table(evaluation_df)
forgetting_df.groupby("method").forgetting.mean()

## 6. Same-source paired trace collection

Every row retains `source_index`. Primary geometry scores are computed within
each source block, then bootstrapped across source images.

In [ ]:
@torch.no_grad()
def collect_paired_trace(model, method, angles, source_indices):
    model.eval()
    rows = []
    loader = multi_loader(angles, False, source_indices, False)
    for views, labels, factors, source_ids in loader:
        batch_size, angle_count = views.shape[:2]
        flat = views.reshape(-1, *views.shape[2:]).to(DEVICE)
        trace = model(flat)
        assert_finite_trace(trace, f"paired trace {method}")

        z0 = trace.z0.reshape(batch_size, angle_count, -1).cpu().numpy()
        context = trace.context.reshape(batch_size, angle_count, -1).cpu().numpy()
        route = trace.route.reshape(batch_size, angle_count, -1).cpu().numpy()
        z_mm = trace.z_mm.reshape(batch_size, angle_count, -1).cpu().numpy()
        logits = trace.logits.reshape(batch_size, angle_count, -1).cpu().numpy()
        hosts = trace.host_outputs.reshape(
            batch_size, angle_count, trace.host_outputs.shape[1], -1
        ).cpu().numpy()

        for b in range(batch_size):
            for a in range(angle_count):
                rows.append({
                    "method": method,
                    "source_index": int(source_ids[b]),
                    "label": int(labels[b]),
                    "angle": float(factors[b, a]),
                    "prediction": int(np.argmax(logits[b, a])),
                    "z0": z0[b, a],
                    "context": context[b, a],
                    "route": route[b, a],
                    "z_mm": z_mm[b, a],
                    "hosts": hosts[b, a],
                })
    return pd.DataFrame(rows)

trace_tables = []
for method, model in models.items():
    trace_tables.append(
        collect_paired_trace(model, method, all_eval_angles, test_indices)
    )
trace_df = pd.concat(trace_tables, ignore_index=True)
print("paired trace rows:", len(trace_df))

## 7. Source-block and centroid geometry

Primary uncertainty is block-bootstrap over source images. Factor-centroid
geometry is reported as a complementary tie-free view. No global all-pairs
p-value is used as evidence.

In [ ]:
def stack_column(frame, name):
    return np.stack(frame[name].to_numpy())

geometry_rows = []

for method in METHODS:
    sub = trace_df[trace_df.method == method]
    factors = sub.angle.to_numpy(float)
    groups = sub.source_index.to_numpy()

    spaces = {
        "sensory": ("euclidean", stack_column(sub, "z0")),
        "context": ("euclidean", stack_column(sub, "context")),
        "route": ("fisher_rao", stack_column(sub, "route")),
        "synthesis": ("euclidean", stack_column(sub, "z_mm")),
    }

    for space, (metric, reps) in spaces.items():
        grouped = grouped_geometry_scores(
            factors,
            reps,
            groups,
            metric=metric,
        )
        rho_mean, rho_low, rho_high = bootstrap_mean_ci(
            grouped["rho"],
            samples=BOOTSTRAP_SAMPLES,
            seed=SEED,
        )
        stress_mean, stress_low, stress_high = bootstrap_mean_ci(
            grouped["stress"],
            samples=BOOTSTRAP_SAMPLES,
            seed=SEED + 1,
        )
        centroid = centroid_geometry_scores(
            factors,
            reps,
            metric=metric,
        )
        geometry_rows.append({
            "method": method,
            "space": space,
            "source_rho_mean": rho_mean,
            "source_rho_ci_low": rho_low,
            "source_rho_ci_high": rho_high,
            "source_stress_mean": stress_mean,
            "source_stress_ci_low": stress_low,
            "source_stress_ci_high": stress_high,
            "centroid_rho": centroid["rho"],
            "centroid_stress": centroid["stress"],
            "source_blocks": len(grouped["rho"]),
            "status": "development_not_qualified",
        })

geometry_df = pd.DataFrame(geometry_rows)
geometry_df

## 8. Held-out interpolation controls

The model is evaluated on held-out angles for accuracy and NLL. Context
centroids are compared with linear interpolation between neighboring trained
centroids. Route centroids are interpolated in square-root simplex coordinates
and compared with a nearest-trained-route control.

In [ ]:
def centroid_by_angle(frame, column):
    return {
        float(angle): np.stack(group[column].to_numpy()).mean(axis=0)
        for angle, group in frame.groupby("angle")
    }

def linear_neighbors(angle, trained):
    lower = max(value for value in trained if value < angle)
    upper = min(value for value in trained if value > angle)
    alpha = (angle - lower) / (upper - lower)
    return lower, upper, alpha

interpolation_rows = []
final_stage = int(evaluation_df.stage.max())

for method in METHODS:
    final_eval = evaluation_df[
        (evaluation_df.method == method)
        & (evaluation_df.stage == final_stage)
        & (evaluation_df.angle_type == "interpolation")
    ]
    sub = trace_df[trace_df.method == method]
    context_centroids = centroid_by_angle(sub, "context")
    route_centroids = centroid_by_angle(sub, "route")

    for row in final_eval.itertuples():
        angle = float(row.angle)
        lower, upper, alpha = linear_neighbors(angle, train_angles)

        c_pred = (
            (1.0 - alpha) * context_centroids[lower]
            + alpha * context_centroids[upper]
        )
        context_error = float(np.linalg.norm(context_centroids[angle] - c_pred))

        p_low = route_centroids[lower]
        p_high = route_centroids[upper]
        root_pred = (
            (1.0 - alpha) * np.sqrt(np.clip(p_low, 1e-12, None))
            + alpha * np.sqrt(np.clip(p_high, 1e-12, None))
        )
        p_pred = np.square(root_pred)
        p_pred = p_pred / p_pred.sum()

        actual = route_centroids[angle]
        route_interp_error = fisher_rao_distance(actual, p_pred)
        nearest_error = min(
            fisher_rao_distance(actual, p_low),
            fisher_rao_distance(actual, p_high),
        )

        interpolation_rows.append({
            "method": method,
            "angle": angle,
            "accuracy": float(row.accuracy),
            "nll": float(row.nll),
            "context_interpolation_error": context_error,
            "route_interpolation_error": route_interp_error,
            "nearest_route_error": nearest_error,
            "route_interpolation_beats_nearest": route_interp_error < nearest_error,
            "status": "development_not_qualified",
        })

interpolation_df = pd.DataFrame(interpolation_rows)
interpolation_df

## 9. Host ablation bundle

Ablation is summarized by mean, median and positive-contribution fraction.
Route mass alone is not interpreted as specialization.

In [ ]:
@torch.no_grad()
def host_ablation_table(model, method, angles, source_indices):
    model.eval()
    rows = []
    for angle in angles:
        loader = fixed_loader(angle, False, source_indices, False)
        for x, y, _, source_id in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            trace = model(x)
            base = F.log_softmax(trace.logits, -1).gather(
                1, y[:, None]
            ).squeeze(1)
            for host in range(trace.route.shape[1]):
                route = trace.route.clone()
                route[:, host] = 0.0
                route = route / route.sum(1, keepdim=True).clamp_min(1e-8)
                z = torch.einsum("bh,bhd->bd", route, trace.host_outputs)
                logits = model.classifier(model.synthesis_norm(z))
                ablated = F.log_softmax(logits, -1).gather(
                    1, y[:, None]
                ).squeeze(1)
                impact = (base - ablated).detach().cpu().numpy()
                for idx, value in enumerate(impact):
                    rows.append({
                        "method": method,
                        "angle": angle,
                        "source_index": int(source_id[idx]),
                        "host": host,
                        "ablation_impact": float(value),
                    })
    raw = pd.DataFrame(rows)
    summary = (
        raw.groupby(["method", "angle", "host"])
        .ablation_impact
        .agg(
            mean="mean",
            median="median",
            positive_fraction=lambda values: float(np.mean(values > 0)),
            count="count",
        )
        .reset_index()
    )
    return raw, summary

ablation_raw_tables = []
ablation_summary_tables = []
for method, model in models.items():
    raw, summary = host_ablation_table(
        model,
        method,
        train_angles + interp_angles,
        test_indices,
    )
    ablation_raw_tables.append(raw)
    ablation_summary_tables.append(summary)

ablation_raw_df = pd.concat(ablation_raw_tables, ignore_index=True)
ablation_summary_df = pd.concat(ablation_summary_tables, ignore_index=True)
ablation_summary_df.head()

## 10. Signed causal route-direction probe

The probe estimates a context direction associated with increasing angle and a
route direction from route centroids. It reports a **signed** projection of the
route change and compares it with matched-norm context directions orthogonal to
the angle tangent.

This remains development evidence. Qualification requires bootstrap confidence
intervals, class-identity gates and replication across seeds.

In [ ]:
def orthogonal_directions(tangent, count, seed):
    rng = np.random.default_rng(seed)
    directions = []
    for _ in range(count):
        vector = rng.normal(size=tangent.shape)
        vector = vector - np.dot(vector, tangent) * tangent
        norm = np.linalg.norm(vector)
        if norm > 1e-12:
            directions.append(vector / norm)
    return directions

@torch.no_grad()
def causal_probe(model, method, trace, probe_angle=15.0, scales=(-2, -1, 0, 1, 2)):
    trained = trace[trace.angle.isin(train_angles)]
    contexts = stack_column(trained, "context").astype(np.float64)
    angles = trained.angle.to_numpy(float)

    x = contexts - contexts.mean(0, keepdims=True)
    y = angles - angles.mean()
    ridge = 1e-6 * max(np.trace(x.T @ x) / max(x.shape[1], 1), 1.0)
    tangent = np.linalg.solve(
        x.T @ x + ridge * np.eye(x.shape[1]),
        x.T @ y,
    )
    tangent = tangent / max(np.linalg.norm(tangent), 1e-12)

    route_centroids = centroid_by_angle(trained, "route")
    route_matrix = np.stack([
        np.sqrt(np.clip(route_centroids[angle], 1e-12, None))
        for angle in train_angles
    ])
    centered_angles = np.asarray(train_angles) - np.mean(train_angles)
    route_centered = route_matrix - route_matrix.mean(0, keepdims=True)
    route_direction = (
        centered_angles[:, None] * route_centered
    ).sum(axis=0) / max(np.sum(centered_angles ** 2), 1e-12)
    route_direction = route_direction / max(np.linalg.norm(route_direction), 1e-12)

    orthogonals = orthogonal_directions(tangent, count=8, seed=SEED + 9)
    loader = fixed_loader(probe_angle, False, test_indices, False)
    x_batch, y_batch, _, _ = next(iter(loader))
    x_batch, y_batch = x_batch.to(DEVICE), y_batch.to(DEVICE)
    base = model(x_batch)
    base_logp = F.log_softmax(base.logits, -1).gather(1, y_batch[:, None]).squeeze(1)

    tangent_tensor = torch.tensor(tangent, dtype=base.context.dtype, device=DEVICE)
    route_dir_tensor = torch.tensor(
        route_direction, dtype=base.route.dtype, device=DEVICE
    )

    rows = []
    for scale in scales:
        new_context = base.context + float(scale) * tangent_tensor
        new_route = torch.softmax(
            model.router(torch.cat([base.z0, new_context], dim=-1)),
            dim=-1,
        )
        delta_root = torch.sqrt(new_route.clamp_min(1e-12)) - torch.sqrt(
            base.route.clamp_min(1e-12)
        )
        signed_effect = float(
            (delta_root * route_dir_tensor).sum(dim=1).mean().detach().cpu()
        )

        z = torch.einsum("bh,bhd->bd", new_route, base.host_outputs)
        logits = model.classifier(model.synthesis_norm(z))
        new_logp = F.log_softmax(logits, -1).gather(1, y_batch[:, None]).squeeze(1)
        identity_drop = float((base_logp - new_logp).mean().detach().cpu())

        orth_effects = []
        for direction in orthogonals:
            direction_tensor = torch.tensor(
                direction, dtype=base.context.dtype, device=DEVICE
            )
            c_orth = base.context + float(scale) * direction_tensor
            r_orth = torch.softmax(
                model.router(torch.cat([base.z0, c_orth], dim=-1)),
                dim=-1,
            )
            delta_orth = torch.sqrt(r_orth.clamp_min(1e-12)) - torch.sqrt(
                base.route.clamp_min(1e-12)
            )
            orth_effects.append(
                float(
                    torch.abs(
                        (delta_orth * route_dir_tensor).sum(dim=1)
                    ).mean().detach().cpu()
                )
            )

        orth_mean = float(np.mean(orth_effects)) if orth_effects else float("nan")
        csr = abs(signed_effect) / max(orth_mean, 1e-12) if scale != 0 else 0.0
        rows.append({
            "method": method,
            "probe_angle": probe_angle,
            "scale": scale,
            "signed_route_effect": signed_effect,
            "orthogonal_abs_effect_mean": orth_mean,
            "causal_specificity_ratio": csr,
            "class_log_prob_drop": identity_drop,
            "status": "development_not_qualified",
        })
    return pd.DataFrame(rows)

causal_tables = []
for method, model in models.items():
    causal_tables.append(
        causal_probe(
            model,
            method,
            trace_df[trace_df.method == method],
        )
    )
causal_df = pd.concat(causal_tables, ignore_index=True)
causal_df

## 11. Development gate summary

The notebook computes measurements needed for C1-C4, but it does not
automatically convert a development run into a claim. A gate may be marked
`candidate_pass` only after frozen thresholds, multi-seed aggregation and all
required controls are present.

In [ ]:
gate_rows = []

for method in METHODS:
    route_row = geometry_df[
        (geometry_df.method == method) & (geometry_df.space == "route")
    ].iloc[0]
    interp = interpolation_df[interpolation_df.method == method]
    forgetting = forgetting_df[forgetting_df.method == method]

    gate_rows.append({
        "method": method,
        "C1_route_source_rho": route_row.source_rho_mean,
        "C1_route_rho_ci_low": route_row.source_rho_ci_low,
        "C1_route_centroid_rho": route_row.centroid_rho,
        "C2_mean_interpolation_accuracy": interp.accuracy.mean(),
        "C2_route_interp_win_fraction": interp.route_interpolation_beats_nearest.mean(),
        "mean_trained_angle_forgetting": forgetting.forgetting.mean(),
        "qualification_status": "not_qualified",
    })

gate_summary_df = pd.DataFrame(gate_rows)
gate_summary_df

## 12. Export and immutable manifest

All exported tables retain epistemic status. The manifest records the package
version, git SHA, split checksums, compared methods and notebook history.

In [ ]:
output_root = Path("results/v1_0_3") / f"seed_{SEED}"
output_root.mkdir(parents=True, exist_ok=True)

stage_df.to_csv(output_root / "stage_logs.csv", index=False)
evaluation_df.to_csv(output_root / "staged_evaluation.csv", index=False)
forgetting_df.to_csv(output_root / "forgetting.csv", index=False)
geometry_df.to_csv(output_root / "paired_geometry_metrics.csv", index=False)
interpolation_df.to_csv(output_root / "interpolation_metrics.csv", index=False)
ablation_summary_df.to_csv(output_root / "host_ablation_summary.csv", index=False)
ablation_raw_df.to_csv(output_root / "host_ablation_raw.csv", index=False)
causal_df.to_csv(output_root / "causal_probe.csv", index=False)
gate_summary_df.to_csv(output_root / "development_gate_summary.csv", index=False)
pd.DataFrame(sensory_history).to_csv(output_root / "sensory_history.csv", index=False)

run_manifest = {
    "status": "development_not_qualified",
    "notebook_version": "1.0.3",
    "package_version": package_version,
    "git_sha": git_sha,
    "run_mode": RUN_MODE,
    "seed": SEED,
    "device": str(DEVICE),
    "methods": METHODS,
    "train_angles": train_angles,
    "interpolation_angles": interp_angles,
    "extrapolation_angles": extra_angles,
    "split_manifest": split_manifest,
    "tracked_changes": [f"CHG-103-{index:02d}" for index in range(1, 11)],
    "warning": (
        "This archive implements the corrected paired protocol. "
        "It is not a qualified C1-C6 result."
    ),
    "timestamp": time.time(),
}
(output_root / "run_manifest.json").write_text(
    json.dumps(run_manifest, indent=2),
    encoding="utf-8",
)
print("Exported to:", output_root.resolve())

## 13. Qualification checklist

Before any C1-C6 claim:

1. Freeze gates before qualification runs.
2. Run at least five seeds for pilot qualification and ten for final evidence.
3. Validate `no_geometry` and `paired_geometry` with equal budgets.
4. Add tuned EWC, replay, sparse-MoE, oracle-angle and joint upper-bound controls.
5. Report source-block confidence intervals and failed seeds.
6. Confirm interpolation angles never enter training or checkpoint selection.
7. Add role matching across seeds for host specialization.
8. Add memory-transport and dual-memory experiments before C5.
9. Replicate on a secondary controlled factor or dataset.
10. Archive notebook, CSVs, manifest, environment and commit SHA.

## 14. Optional PDF export

The GitHub archive contains a separate protocol correction report under
`docs/reports/`. Notebook PDF export is optional and should not replace the raw
`.ipynb`, CSV tables or manifest.

In [ ]:
# Optional: save/upload this notebook first, then run.
# The command passes each list-valued latex argument separately.
#
# import subprocess, sys
# from pathlib import Path
#
# notebook_path = Path("/content/Geometry_MMALS_G1_CrossAngle_Paired_v1_0_3.ipynb")
# output_dir = Path("/content/pdf_export")
# output_dir.mkdir(parents=True, exist_ok=True)
# subprocess.run([
#     sys.executable, "-m", "jupyter", "nbconvert",
#     "--to", "pdf",
#     str(notebook_path),
#     "--output-dir", str(output_dir),
#     "--PDFExporter.latex_command=xelatex",
#     "--PDFExporter.latex_command={filename}",
#     "--PDFExporter.latex_command=-interaction=nonstopmode",
#     "--PDFExporter.latex_count=3",
# ], check=True)